<a href="https://colab.research.google.com/github/babayaga477/BABAYAGA/blob/main/Inteviewer_Co_Pilot_Gen_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [79]:
#Install required libraries
!pip install python-docx PyPDF2 pdfplumber google-generativeai


In [80]:
#Imports
from google.colab import files
import os
from docx import Document
import PyPDF2
import pdfplumber
import re
import time
import json
import google.generativeai as genai


In [81]:
#File Upload
def upload_file():
    """Handles file upload in Colab"""
    uploaded = files.upload()
    return list(uploaded.keys())[0] if uploaded else None


In [82]:
#Text Extraction
def extract_text(file_path):
    ext = file_path.lower()
    if ext.endswith(".pdf"):
        with pdfplumber.open(file_path) as pdf:
            return "\n".join([page.extract_text() for page in pdf.pages if page.extract_text()])
    elif ext.endswith(".docx"):
        doc = Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs if para.text])
    else:
        raise ValueError("Unsupported file format")


In [83]:
#Gemini Initialization
def initialize_gemini(api_key):
    """Initialize Gemini model"""
    genai.configure(api_key=api_key)
    return genai.GenerativeModel('gemini-2.0-flash')


In [84]:
#Project Extraction via Gemini
def extract_projects(text, model):
    safe_text = text[:20000]
    prompt = f"""Extract technical projects from this CV with:
- Title: Clear project name
- Tech Stack: Comma-separated technologies
- Description: 2-3 sentence summary

Format each project like this:
=== PROJECT ===
Title: [name]
Tech Stack: [technologies]
Description: [summary]
=== END ===

CV Content:
{safe_text}
"""

    try:
        response = model.generate_content(prompt)
        return parse_projects(response.text)
    except Exception as e:
        print("Gemini error:", e)
        return []


In [85]:
#Parse Gemini Output
def parse_projects(response_text):
    projects = []
    for block in response_text.split("=== PROJECT ===")[1:]:
        try:
            project = {
                "title": "N/A",
                "tech_stack": "N/A",
                "description": "N/A"
            }
            lines = [l.strip() for l in block.split("=== END ===")[0].split("\n") if l.strip()]
            for line in lines:
                if re.match(r"Title\s*:", line, re.I):
                    project["title"] = line.split(":", 1)[1].strip()
                elif re.match(r"Tech Stack\s*:", line, re.I):
                    project["tech_stack"] = line.split(":", 1)[1].strip()
                elif re.match(r"Description\s*:", line, re.I):
                    project["description"] = line.split(":", 1)[1].strip()
            if project["title"] != "N/A":
                projects.append(project)
        except:
            continue
    return projects


In [86]:

api_key=""
file_path=""
projects={}
#Main Function
def process_cv_to_json():
    global api_key
    global file_path
    global projects
    file_path = upload_file()
    raw_text = extract_text(file_path)
    print("\n📄 Extracted CV Text (preview):\n")
    print(raw_text[:1000], "\n...")

    # Initialize Gemini
    api_key = input("🔑 Enter your Gemini API key: ")
    model = initialize_gemini(api_key)

    # Get structured project data
    projects = extract_projects(raw_text, model)
    final_output = {"projects": projects}

    # Display result
    print("\n📦 Structured JSON Output:\n")
    print(json.dumps(final_output, indent=2))
    return final_output


In [91]:
#Upload the file for Text Extraction
output_json = process_cv_to_json()

Saving random_cv.pdf to random_cv (11).pdf



📄 Extracted CV Text (preview):

IoT Engineer Resume Sample
Justin
IoT Engineer
Summary
Having real enthusiasm, I want to build up my career in a prominent and progressive organization that can take full advantage
of my comprehensive knowledge and skills and thus offers career growth opportunities through proven performance in my
knowledge, skill, and effort in the effective field related to works.
Skills
 Simulation Related Software
 HTML
 MS Word, MS Excel, and MS Power Point
 C Programing
 Internet Operations
 Autocad
 Php
 Java Script
Work Experience
Senior Software Engineer
Deloitte Digital
Software Engineer
Accenture LLC
Intern
Ace Technologies
Projects
Automatic Street Light
Leader
Build an ‘automatic street light with LDR’. This circuit employed the output from an uncomplicated light/dark activated circuit
and oblige a relay in its output which can be further attached to switch ON/OFF a street light and electrical application in a
household also.
Digital Logic Design
Pr

Question Generation

In [93]:
#Question Generation
def generate_questions(project, model):
    """Generates interview questions for a single project"""
    prompt = f"""Generate 3 technical interview questions about this project:

    Project: {project['title']}
    Technologies: {project['tech_stack']}
    Details: {project['description']}

    Requirements:
    1. One question about technical implementation
    2. One question about problem-solving
    3. One question about outcomes/learnings
    Format as Python list: ["Q1", "Q2", "Q3"]"""

    try:
        response = model.generate_content(prompt)
        #print(response.text[10:-4])
        return eval(response.text[10:-4])
    except:
        return ["Could not generate questions"]

In [89]:
#Print Questions and Download
def process_cv(api_key):
    """Complete processing pipeline with print output in Colab and downloadable JSON"""
    model = initialize_gemini(api_key)
    if not projects:
        print("No projects found")
        return None

    # 4. Generate questions with error handling
    results = {"filename": file_path, "projects":[]}
    for i, project in enumerate(projects):
        try:
            questions = generate_questions(project, model)
            results["projects"].append({
                "project_info": project,
                "questions": questions
            })
            if i < len(projects) - 1:
                time.sleep(1.5)  # Rate limiting
        except Exception as e:
            print(f"Failed to process project {i+1}: {str(e)}")
            continue

    # 5. Save and return results
    with open("interview_questions.json", "w") as f:
        json.dump(results, f, indent=2)

    # Print the JSON directly in Colab
    print("\n📦 Structured JSON Output (Interview Questions):\n")
    print(json.dumps(results, indent=2))

    # Download the JSON file
    files.download("interview_questions.json")

    return results


In [90]:
final_results = process_cv(api_key)
if final_results:
    print("Successfully processed CV!")
    print(f"Found {len(final_results['projects'])} projects")
else:
    print("Processing failed")


📦 Structured JSON Output (Interview Questions):

{
  "filename": "random_cv (10).pdf",
  "projects": [
    {
      "project_info": {
        "title": "Automatic Street Light",
        "tech_stack": "LDR, Relay",
        "description": "Built an automatic street light using an LDR to activate a relay. The relay switches ON/OFF a street light or other electrical applications in a household based on light/dark conditions."
      },
      "questions": [
        "Describe in detail the circuit you used, specifically how the LDR is connected to the relay and what components are necessary to protect the microcontroller (if any) and ensure the relay switches reliably. Include values of resistors and any relevant transistor specifications.",
        "Suppose the street light flickers intermittently when the light level is near the threshold for switching. How would you troubleshoot this issue, and what modifications could you implement in your hardware or software to prevent this flickering? C

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully processed CV!
Found 7 projects
